In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Load featurised datasets
df_train_feat = pd.read_csv("../data/bbb_train_features.csv")
df_test_feat = pd.read_csv("../data/bbb_test_features.csv")
df_val_feat = pd.read_csv("../data/bbb_valid_features.csv")

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
print("\nOverall Dataset Info:")
df_train_feat.info()


Overall Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1421 entries, 0 to 1420
Columns: 205 entries, Drug_ID to qed
dtypes: float64(200), int64(1), object(4)
memory usage: 2.2+ MB


In [3]:
# Display first few rows
df_train_feat.head()

,Drug_ID,Drug,Y,key,input,BalabanJ,BertzCT,Chi0,Chi0n,Chi0v,...,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,qed
0,Terbutylchlorambucil,CC(C)(C)OC(=O)CCCc1ccc(N(CCCl)CCCl)cc1,1,Terbutylchlorambucil,CC(C)(C)OC(=O)CCCc1ccc(N(CCCl)CCCl)cc1,2.458834,462.905084,17.294682,14.278788,15.790646,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.474821
1,brosotamide,Cc1cc(Br)cc(C(N)=O)c1O,1,brosotamide,Cc1cc(Br)cc(C(N)=O)c1O,3.373859,336.074900,9.300965,6.465477,8.051474,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.768365
2,butacetin,CC(=O)Nc1ccc(OC(C)(C)C)cc1,1,butacetin,CC(=O)Nc1ccc(OC(C)(C)C)cc1,2.681702,335.634906,11.474691,9.625898,9.625898,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.809397
3,Salicyluricacid,O=C(O)CNC(=O)c1ccccc1O,1,Salicyluricacid,O=C(O)CNC(=O)c1ccccc1O,2.700084,361.584516,10.552042,7.227432,7.227432,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.642871
4,sumacetamol,CSCC[C@H](NC(C)=O)C(=O)Oc1ccc(NC(C)=O)cc1,1,sumacetamol,CSCC[C@H](NC(C)=O)C(=O)Oc1ccc(NC(C)=O)cc1,2.667080,530.848914,16.535169,12.842206,13.658703,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.590102


In [4]:
# Get list of column names
columns_list = df_train_feat.columns.tolist()
print(columns_list)

['Drug_ID', 'Drug', 'Y', 'key', 'input', 'BalabanJ', 'BertzCT', 'Chi0', 'Chi0n', 'Chi0v', 'Chi1', 'Chi1n', 'Chi1v', 'Chi2n', 'Chi2v', 'Chi3n', 'Chi3v', 'Chi4n', 'Chi4v', 'EState_VSA1', 'EState_VSA10', 'EState_VSA11', 'EState_VSA2', 'EState_VSA3', 'EState_VSA4', 'EState_VSA5', 'EState_VSA6', 'EState_VSA7', 'EState_VSA8', 'EState_VSA9', 'ExactMolWt', 'FpDensityMorgan1', 'FpDensityMorgan2', 'FpDensityMorgan3', 'FractionCSP3', 'HallKierAlpha', 'HeavyAtomCount', 'HeavyAtomMolWt', 'Ipc', 'Kappa1', 'Kappa2', 'Kappa3', 'LabuteASA', 'MaxAbsEStateIndex', 'MaxAbsPartialCharge', 'MaxEStateIndex', 'MaxPartialCharge', 'MinAbsEStateIndex', 'MinAbsPartialCharge', 'MinEStateIndex', 'MinPartialCharge', 'MolLogP', 'MolMR', 'MolWt', 'NHOHCount', 'NOCount', 'NumAliphaticCarbocycles', 'NumAliphaticHeterocycles', 'NumAliphaticRings', 'NumAromaticCarbocycles', 'NumAromaticHeterocycles', 'NumAromaticRings', 'NumHAcceptors', 'NumHDonors', 'NumHeteroatoms', 'NumRadicalElectrons', 'NumRotatableBonds', 'NumSaturat

# **XGBoost Model Training for Molecular Permeability Prediction**

## **Project Overview**
This notebook trains an **XGBoost classification model** to predict whether a molecule is **permeable (Yes/1) or non-permeable (No/0)** based on its **physicochemical descriptors**.  

**Dataset Information:**  
- The dataset consists of molecular descriptors derived from **SMILES strings**.
- Features include **BalabanJ, BertzCT, ExactMolWt, NumHAcceptors**, and more.
- The **target variable (`Y`)** is binary:  
  - `1` → Permeable  
  - `0` → Non-permeable  
- We already have **preprocessed train, test, and validation datasets**.

---

# **Data Preparation**
Before training the model, we **separate features from labels** and remove columns that are not useful for training.  

In [6]:
# Define the target variable (label)
target_col = "Y"  # Assuming "Y" is the label column

# Define columns to drop (non-feature columns)
non_feature_cols = ["Drug_ID", "Drug", "key", "input", "Y"]  

# Extract features (X) and labels (y) for training, testing, and validation
X_train = df_train_feat.drop(columns=non_feature_cols)
y_train = df_train_feat[target_col]

X_test = df_test_feat.drop(columns=non_feature_cols)
y_test = df_test_feat[target_col]

X_val = df_val_feat.drop(columns=non_feature_cols)
y_val = df_val_feat[target_col]


### **Why This Step is Important?**
- `Drug_ID`, `Drug`, and `key` are **identifiers** and do not contribute to predictions.
- `input` might be a **text-based feature (e.g., SMILES string)**, which is not used directly.
- Dropping these ensures that the model **only learns from numerical descriptors**.

# **Handling Imbalanced Data**
Our dataset has **more permeable (`1`) molecules than non-permeable (`0`)**.  
To **prevent bias**, we use **SMOTE (Synthetic Minority Over-sampling Technique)** to **balance the training data**.

In [9]:
pip install --upgrade scikit-learn imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
os.environ["SCIPY_ARRAY_API"] = "1"

# Then proceed with imports
from imblearn.over_sampling import SMOTE
from sklearn.utils.validation import check_X_y

# Validate X_train and y_train
X_train_valid, y_train_valid = check_X_y(X_train, y_train)

# Apply SMOTE to balance the dataset
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_valid, y_train_valid)

### **Why This Step is Important?**
- If the dataset is **imbalanced**, the model may **favor the majority class (Yes/1)**.
- SMOTE **generates synthetic samples** for the minority class (`No/0`), **helping the model generalize better**.

# **Hyperparameter Tuning with GridSearchCV**

We'll use **GridSearchCV** to search over a range of hyperparameters for **XGBoost**. This process helps find the best combination of parameters for better model performance.

In [13]:
from sklearn.model_selection import GridSearchCV
import xgboost as xgb

# Define the XGBoost model
xgb_model = xgb.XGBClassifier(random_state=42)

# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.3],
    'max_depth': [3, 6, 9],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

# Set up GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, 
                           scoring='accuracy', cv=5, verbose=1, n_jobs=-1)

# Fit GridSearchCV to the balanced data
grid_search.fit(X_train_balanced, y_train_balanced)

# Best parameters and best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"Best Hyperparameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best Hyperparameters: {'colsample_bytree': 0.7, 'learning_rate': 0.1, 'max_depth': 9, 'n_estimators': 100, 'subsample': 0.7}
Best Cross-Validation Accuracy: 0.9054


### **Why Hyperparameter Tuning?**
- Hyperparameters like `learning_rate`, `max_depth`, and `n_estimators` greatly affect model performance.
- **GridSearchCV** automates the process of testing multiple combinations of hyperparameters to **find the best ones** for our dataset.
- **Cross-validation** is used to **evaluate the model's performance** across different folds, ensuring a robust choice of hyperparameters.